In [1]:
import pandas as pd
from datasets import load_dataset, DatasetDict, Dataset, concatenate_datasets

/Users/seankim/Documents/GitHub/ChessPuzzleEvaluator/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_dataset("Lichess/chess-puzzles")

In [3]:
DATASET_SIZE = 1000000
subset = ds["train"].shuffle(seed=42).select(range(DATASET_SIZE))

In [4]:
subset.features

{'PuzzleId': Value('large_string'),
 'GameId': Value('string'),
 'FEN': Value('large_string'),
 'Moves': Value('large_string'),
 'Rating': Value('uint16'),
 'RatingDeviation': Value('uint16'),
 'Popularity': Value('int8'),
 'NbPlays': Value('uint32'),
 'Themes': List(Value('string')),
 'OpeningTags': List(Value('string'))}

In [5]:
subset[0]

{'PuzzleId': '1ttUB',
 'GameId': 'XL6z9NwE#19',
 'FEN': 'r2qkb1r/pppb1ppp/4p3/8/1n6/1PN2Q2/1PPPNPPP/R1BK3R w kq - 1 10',
 'Moves': 'f3b7 d7c6 b7c6 b4c6',
 'Rating': 1344,
 'RatingDeviation': 76,
 'Popularity': 90,
 'NbPlays': 1728,
 'Themes': ['crushing', 'master', 'opening', 'short', 'trappedPiece'],
 'OpeningTags': ['Alekhine_Defense',
  'Alekhine_Defense_Scandinavian_Variation']}

In [6]:
len(subset)

1000000

In [7]:
# train test split
train_test = subset.train_test_split(test_size=0.2, seed=42)
test_val = train_test["test"].train_test_split(test_size=0.5, seed=42)
ds_dict = DatasetDict({
    "train": train_test["train"],
    "val": test_val["train"],
    "test": test_val["test"],
})

In [8]:
print("train samples", len(ds_dict["train"]), "val samples", len(ds_dict["val"]), "test samples", len(ds_dict["test"]))

train samples 800000 val samples 100000 test samples 100000


### Feature Augmentation

In [9]:
import chess
import chess.engine
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from sklearn.preprocessing import MultiLabelBinarizer

Potential Features:
- theme_*, has mate theme
- mate in n
- num moves in solution
- puzzle rating + rating deviation
- is check
- legal move count
- figure move is quiet
- first move is sacrifice
- has underpromotion
- total material on board
- material balance abs
- hanging pieces (white + black)
- passed pawns (white & black)

In [10]:
ALL_THEMES = set([item for sublist in subset["Themes"] for item in sublist])
MATE_THEMES = {'mate', 'mateIn1', 'mateIn2', 'mateIn3', 'mateIn4', 'mateIn5',}
PIECE_VALUES = {chess.PAWN:100, chess.KNIGHT:300, chess.BISHOP:300, chess.ROOK:500, chess.QUEEN:900}

In [11]:
# General
def num_moves(moves: str) -> int:
    return max(0, len(moves.split()))

def num_openings(openings: str) -> int:
    return len(set(openings)) if openings is not None else 0

# Themes
def has_mate(themes: list[str]) -> bool:
    return bool(set(themes) & MATE_THEMES)

def num_themes(themes: list[str]) -> int:
    return len(set(themes))

In [12]:
def calculate_material(board: chess.Board) -> tuple[int, int]:
    w_mat = sum(PIECE_VALUES[p.piece_type] for p in board.piece_map().values() if p.color == chess.WHITE and p.piece_type in PIECE_VALUES)
    b_mat = sum(PIECE_VALUES[p.piece_type] for p in board.piece_map().values() if p.color == chess.BLACK and p.piece_type in PIECE_VALUES)
    return w_mat, b_mat

def calculate_hanging(board: chess.Board, is_white: bool) -> int:
    color = chess.WHITE if is_white else chess.BLACK
    return sum(1 for sq, p in board.piece_map().items() if p.color == color and p.piece_type != chess.KING and board.attackers(not color, sq) and not board.attackers(color, sq))

In [13]:
def extract_features(fen, moves, themes, openings):
    f = {}
    move_list = moves.split()

    # General
    f["num_moves"] = num_moves(moves)
    f["num_openings"] = num_openings(openings)

    # Themes
    f["num_themes"] = num_themes(themes)
    f["has_mate"] = has_mate(themes)
    
    # board
    board = chess.Board(fen)
    w_mat, b_mat = calculate_material(board)
    f["total_material"] = w_mat + b_mat
    f["material_balance"] = abs(w_mat - b_mat)
    f["is_endgame"] = int((w_mat + b_mat) < 2600)
    f["is_check"] = int(board.is_check())
    f["legal_move_count"] = int(board.legal_moves.count())
    f["white_hanging"] = calculate_hanging(board, True)
    f["black_hanging"] = calculate_hanging(board, False)

    # solution first move
    if len(move_list) >= 2:
        board.push(chess.Move.from_uci(move_list[0]))
        mv = chess.Move.from_uci(move_list[1])
        moving = board.piece_at(mv.from_square)
        captured = board.piece_at(mv.to_square)
        is_cap = board.is_capture(mv)
        board.push(mv)

        f["first_move_is_capture"] = int(is_cap)
        f["first_move_is_check"] = int(board.is_check())
        f["first_move_is_quiet"] = int(not is_cap and not board.is_check())
        f["first_move_is_sacrifice"] = int(bool(is_cap and moving and captured and PIECE_VALUES.get(moving.piece_type, 0) > PIECE_VALUES.get(captured.piece_type, 0)))
        f["has_underpromotion"] = int(any(chess.Move.from_uci(m).promotion not in (None, chess.QUEEN) for m in move_list[1:]))

    return f


In [14]:
def extract_features_row(row):
    return extract_features(row["FEN"], row["Moves"], row["Themes"], row["OpeningTags"])

new_subset = subset.map(extract_features_row, batched=False, num_proc=4)

Map (num_proc=4): 100%|██████████| 1000000/1000000 [00:44<00:00, 22247.52 examples/s]


In [15]:
# Theme One Hot Encoding
mlb = MultiLabelBinarizer(classes=list(ALL_THEMES))
mlb.fit([ALL_THEMES])

theme_list = [t for t in new_subset["Themes"]]
encoded = mlb.fit_transform(theme_list)
theme_dataset = Dataset.from_dict({
    col: encoded[:, i].tolist()
    for i, col in enumerate(mlb.classes_)
})

new_subset = concatenate_datasets([new_subset, theme_dataset], axis=1)

In [16]:
new_subset[0]

{'PuzzleId': '1ttUB',
 'GameId': 'XL6z9NwE#19',
 'FEN': 'r2qkb1r/pppb1ppp/4p3/8/1n6/1PN2Q2/1PPPNPPP/R1BK3R w kq - 1 10',
 'Moves': 'f3b7 d7c6 b7c6 b4c6',
 'Rating': 1344,
 'RatingDeviation': 76,
 'Popularity': 90,
 'NbPlays': 1728,
 'Themes': ['crushing', 'master', 'opening', 'short', 'trappedPiece'],
 'OpeningTags': ['Alekhine_Defense',
  'Alekhine_Defense_Scandinavian_Variation'],
 'num_moves': 4,
 'num_openings': 2,
 'num_themes': 5,
 'has_mate': False,
 'total_material': 7000,
 'material_balance': 0,
 'is_endgame': 0,
 'is_check': 0,
 'legal_move_count': 41,
 'white_hanging': 0,
 'black_hanging': 1,
 'first_move_is_capture': 0,
 'first_move_is_check': 0,
 'first_move_is_quiet': 1,
 'first_move_is_sacrifice': 0,
 'has_underpromotion': 0,
 'queensideAttack': 0,
 'morphysMate': 0,
 'queenEndgame': 0,
 'mateIn4': 0,
 'oneMove': 0,
 'mateIn2': 0,
 'triangleMate': 0,
 'pin': 0,
 'long': 0,
 'defensiveMove': 0,
 'clearance': 0,
 'discoveredAttack': 0,
 'vukovicMate': 0,
 'mate': 0,
 'knig

In [18]:
dataset = new_subset.remove_columns(["PuzzleId", "GameId", "FEN", "Moves", "Themes", "OpeningTags"])
dataset.to_parquet("puzzles_features.parquet")

Creating parquet from Arrow format: 100%|██████████| 7/7 [00:01<00:00,  5.73ba/s]


689125006